# Strands AgentでOpenAIモデルを使用する

## 概要

Strands Agentsは、モデル駆動型のアプローチでAIエージェントを構築・実行するSDKで、数行のコードで実現できます。Strandsは複数のプロバイダーとどこでもホストされたモデルをサポートします。

[LiteLLM](https://docs.litellm.ai/docs/) は、さまざまなLLMプロバイダーの統一インターフェースで、Amazon、Anthropic、OpenAIなどからのモデルを単一のAPIで操作できます。Strands Agent SDKはLiteLLMプロバイダーを実装しており、LiteLLMがサポートする任意のモデルに対してエージェントを実行できます。

この例では、Microsoft Azureでホストされた `gpt-4.1-mini` モデルをStrands Agentの基盤モデルとして使用する方法を示します。気象と時刻取得ツールを使用したシンプルなエージェントユースケースを使用します。

## Agent Details

<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |LiteLLM model                                      |
|Agent Structure     |Single agent architecture                          |

</div>

## アーキテクチャ

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## 主な機能

* **LiteLLM model**: LiteLLM経由で提供されるモデルを使用

## セットアップと前提条件

### 前提条件

* Python 3.10+

* Azure Account

* gpt-4.1-mini access

Strands Agentの要件パッケージをインストールしましょう

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### 依存パッケージのインポート

依存パッケージをインポートしましょう

In [ ]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.litellm import LiteLLMModel

### Azureキーの設定

Azure APIキーを設定しましょう

In [ ]:
os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

### カスタムツールの設定

エージェントをテストするための2つのダミーツールを設定しましょう

In [ ]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Please replace with actual weather API call.
    return "sunny"

### エージェントの基盤LLMモデルの定義

次に、LiteLLMを使用してエージェントの基盤モデルを定義します。`gpt-4.1-mini` に設定します

In [ ]:
model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)

### エージェントの定義

必要な情報がすべて揃ったので、エージェントを定義しましょう

In [ ]:
system_prompt = "You are a simple agent that can tell the time and the weather"
agent = Agent(
    model=litellm_model,
    system_prompt=system_prompt,
    tools=[current_time, current_weather],
)

### エージェントのテスト

エージェントを呼び出してテストしましょう

In [ ]:
results = agent("What time is it in Seattle? And how is the weather?")

#### エージェントの結果の分析

素晴らしい！初めてエージェントを呼び出しました！resultsオブジェクトを探索しましょう。最初に見られるのは、エージェントのオブジェクトで交換されるメッセージです

In [ ]:
agent.messages

次に、result `metrics` を分析することで、最後のクエリに対するエージェントの使用状況を確認できます

In [ ]:
results.metrics

### おめでとうございます！

このノートブックでは、気象エージェントの回答を提供するOpenAIでLiteLLMを使用する方法を学びました。